[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/14_evaluacion_online/14_proyecto_evaluacion_online.ipynb)

# Proyecto 14 · Evaluador online de SupportOps (y su panel en Grafana)

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Ninguno (se evalúa en memoria). Opcional: `core` + `metrics` para ver tu panel (paso final) |
| **RAM aproximada** | < 500 MB |
| **Prerrequisitos** | Lección `14_evaluacion_online`, módulo 10 (dashboards como código) y proyecto 13 |
| **Conexión** | Hasta ahora SupportOps medía *si responde*; hoy mide *si responde bien*. El 15 usará tu evaluador para cazar regresiones y el 16 lo convertirá en un SLO |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. Producto sospecha que el agente «se inventa cosas» y no hay forma de saberlo: todos los paneles están en verde. Te piden un
**evaluador online** con cuatro garantías: (1) detecta respuestas sin respaldo, (2) cuesta poco porque **muestrea**, (3) su número poblacional **no está sesgado** y (4) se ve en Grafana y es
auditable (cada nota cuelga de la traza evaluada). Además, tu jefe quiere saber **cuánto se puede fiar del juez**.

## Datos

Sintéticos y deterministas (semilla 42, `mock`): el **Paso 0** ejecuta 600 tickets con alucinaciones al 6 % y construye un `EvalCase` por ticket. Los primeros 200 forman el **conjunto de calibración**
(etiquetado por «humanos» = la verdad del inyector); los 400 restantes son «producción» sin etiquetas. Cadena de datos: stack local (opcional) → memoria.

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución; `mock`, semilla 42) |
|---|---|---|---|
| R1 | `check_groundedness` | Sensibilidad y especificidad frente a la verdad de los 600 casos | ≥ 0,98 ambas |
| R2 | `stratified_estimate` | Estimador post-estratificado y error estándar sobre una muestra dada | exacto (tol. 1e-9) |
| R3 | `eval_telemetry` | Evento válido contra la spec (instantánea) + etiquetas de métrica acotadas | 5/5 comprobaciones |
| R4 | `bias_probe` | Ganancia por verbosidad y consistencia de posición del juez | ± 0,02 / ± 0,05 de la referencia |
| R5 | `calibrate` | Sensibilidad/especificidad con IC de Wilson y corrección de la tasa en producción | error ≤ 0,01 frente a la verdad |
| R6 | `choose_rate` | Mayor tasa de muestreo que cumple el presupuesto de coste y su error de estimación | exacto |
| R7 | `build_quality_dashboard` | `lint` limpio y 8 consultas clave | 8/8 |

> 📏 Los umbrales salen de la solución de referencia: R1 se comprobó con las semillas 42, 7 y 99 (sensibilidad y especificidad = 1,0 en las tres); R2, R4 y R5 son cifras exactas de la semilla 42, fija en el notebook. Ojo con R5: la calibración tiene solo ≈ 15 alucinaciones positivas, así que el IC de la sensibilidad es **ancho** (la corrección de Rogan–Gladen con otra semilla puede quedar peor que la tasa observada: justo la lección). El tiempo es simulado: nada depende de tu máquina.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import dataclasses, json, math, re, time
from supportops.instrument import CompositeInstrumentor
from supportops.telemetry import OTelInstrumentor, setup_telemetry
from supportops.quality_eval import (EVALUATORS, EVENT_NAME, CaseCollector, EvalCase, EvalScore, LLMJudge, MockJudge, OnlineEvaluator, RateSampler, RuleJudge,
                                     StratifiedSampler, build_eval_event, confusion, extract_claims, find_pii, post_stratified_rate, render_judge_prompt,
                                     required_sample_size, rogan_gladen, scrub_pii, simulate_feedback, unsupported_claims, wilson_interval)
STACK = check_stack([])
LIVE = False                      # True: Collector + Prometheus responden y la sección 8 envía telemetría de verdad

from supportops.dashkit import Dashboard, lint, prom, loki, tempo, timeseries, stat, bargauge, traces_table, logs, table, heatmap, thresholds

## Paso 0 · Los datos (te lo damos hecho)

In [ ]:
N = 600
FAULTS = {**DEFAULT_FAULT_RATES, "hallucination": 0.06}
TK = generate_tickets(N, seed=SEED, fault_rates=FAULTS)
_col = CaseCollector()
RES = run_tickets(SupportAgent(instrumentor=_col), TK)
CASES = _col.cases
JUDGE, RULE = MockJudge(SEED), RuleJudge()
CALIB, PROD = CASES[:200], CASES[200:]                              # calibración (etiquetada por humanos) / producción (sin etiquetas)
TRUTH = [not c.truth["grounded"] for c in CASES]                    # verdad: ¿alucinada?

# muestra estratificada ya evaluada (por el juez de reglas) para el Paso 2
_ev = OnlineEvaluator(RULE, StratifiedSampler(salt="proyecto"))
for c in CASES: _ev.submit(c)
_ev.run_pending()
EVALS = [{"stratum": r.stratum, "passed": bool(r.verdict.passed)} for r in _ev.records]
POP = dict(_ev.population)
FULL_PASS = float(np.mean([RULE.evaluate(c).passed for c in CASES]))      # lo que daría evaluar TODO
print(f"{len(CASES)} casos · alucinadas: {sum(TRUTH)} · muestra evaluada: {len(EVALS)} · población por estrato: {POP}")
print(f"calidad OK real (evaluando todo): {FULL_PASS:.3f} · media ingenua de la muestra: {np.mean([e['passed'] for e in EVALS]):.3f}")

## Paso 1 · Un evaluador de groundedness hecho por ti

**TODO 1.** `check_groundedness(answer, context, question)` devuelve `{"score": 0.0 | 1.0, "unsupported": [(tipo, valor), …]}`. La respuesta está **fundamentada** (`score=1.0`) si toda afirmación verificable está respaldada:

- **Afirmaciones**: identificadores `ORD-\d{5}`, `TRK\d{6,}`, `RF-\d+`, `HO-\d+`, `KB-\d+`; importes `NN,NN €`; y «número + unidad» con unidad ∈ {años, unidades, horas, meses, días}.
- **Respaldo**: un identificador está respaldado si aparece literalmente en `context` o en `question`; un importe o «número + unidad», si su **número** (con `,` o `.` decimal) aparece entre los números del `context`.
- **Excepciones** (no son afirmaciones): el ejemplo de formato `ORD-00000` y los hechos de política **«30 días»** y **«5 días»**.

<details>
<summary>🪜 Pista 1</summary>

Usa `re.findall` para cada familia de afirmaciones y normaliza los números con `float(x.replace(',', '.'))`. Para «número + unidad» captura `(\d+)\s+(años|unidades|horas|meses|días|dias)`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`nums = {round(float(x.replace(',', '.')), 2) for x in re.findall(r'\d+(?:[.,]\d+)?', context)}`; un importe/num está respaldado si `round(valor, 2) in nums`. Descarta primero `ORD-00000` y los textos de política (`'30 días'`, `'5 días'`).

</details>

In [ ]:
def check_groundedness(answer: str, context: str, question: str = "") -> dict:
    # TODO 1
    raise NotImplementedError("TODO 1")

try:
    print(check_groundedness("Reembolso aprobado de 1299,00 € (ref. RF-99999).", '{"order_id": "ORD-00001", "total_eur": 129.0}', "quiero un reembolso de ORD-00001"))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · Estimar la calidad poblacional sin sesgo

**TODO 2.** `stratified_estimate(evals, population)` recibe la lista `EVALS` (dicts `stratum`, `passed`) y la población por estrato `POP`, y devuelve `(estimación, error_estándar)` con el estimador **post-estratificado** de la lección (sección 6):
estimación \( \sum_s \frac{N_s}{N}\hat p_s \), varianza \( \sum_s (N_s/N)^2\,\hat p_s(1-\hat p_s)/(n_s-1)\,(1-n_s/N_s) \). Ignora los estratos sin evaluaciones y usa `max(n_s - 1, 1)` en el denominador.

<details>
<summary>🪜 Pista 1</summary>

Agrupa `evals` por estrato: `n_s` = nº de evaluaciones, `k_s` = nº de `passed`. Después recorre `population.items()`.

</details>

In [ ]:
def stratified_estimate(evals: list, population: dict) -> tuple:
    # TODO 2
    raise NotImplementedError("TODO 2")

try:
    est, se = stratified_estimate(EVALS, POP)
    print(f"estimación {est:.4f} ± {1.96 * se:.4f} · media ingenua {np.mean([e['passed'] for e in EVALS]):.4f} · verdad {FULL_PASS:.4f}")
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 3 · Publicar la evaluación como telemetría

**TODO 3.** `eval_telemetry(case, name, score, label, explanation, stratum, sample_rate)` devuelve `(event_attrs, metric_attrs)`:

- `event_attrs`: atributos del evento `gen_ai.evaluation.result`: `gen_ai.evaluation.name`, `gen_ai.evaluation.score.value` (**float**), `gen_ai.evaluation.score.label`, `gen_ai.evaluation.explanation` (recortada a **300** caracteres, solo si hay), `gen_ai.response.id` (solo si el caso lo tiene) y los propios `supportops.ticket.id`, `supportops.eval.stratum`, `supportops.eval.sample_rate`, `gen_ai.agent.version`.
- `metric_attrs`: etiquetas **acotadas** para la métrica `supportops.eval.results`: exactamente `gen_ai.evaluation.name`, `gen_ai.evaluation.score.label`, `supportops.eval.stratum`, `gen_ai.agent.version`. **Nunca** ids, explicación ni puntuación.

In [ ]:
def validate_eval_event(attrs: dict) -> list:
    snap = json.load(open(ROOT / "04_genai_semconv" / "genai_semconv_snapshot.json"))
    probs = []
    if "gen_ai.evaluation.name" not in attrs: probs.append("falta gen_ai.evaluation.name")
    if "gen_ai.evaluation.score.value" not in attrs and "gen_ai.evaluation.score.label" not in attrs: probs.append("falta score.value o score.label")
    for k, v in attrs.items():
        if k.startswith("gen_ai."):
            if k not in snap["attributes"]: probs.append(f"{k} no existe en la spec")
            elif snap["attributes"][k]["type"] == "double" and not isinstance(v, float): probs.append(f"{k} debe ser double")
            elif snap["attributes"][k]["type"] == "string" and not isinstance(v, str): probs.append(f"{k} debe ser string")
    return probs

def eval_telemetry(case, name: str, score: float, label: str, explanation: str, stratum: str, sample_rate: float) -> tuple:
    # TODO 3
    raise NotImplementedError("TODO 3")

try:
    ea, ma = eval_telemetry(CASES[0], "groundedness", 1, "pass", "ok", "base", 0.1)
    print(validate_eval_event(ea) or "evento válido", ma)
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · Sondas de sesgo del juez

**TODO 4.** `bias_probe(judge, cases, pad)` devuelve `{"verbosity_gain": …, "position_consistency": …}` usando **solo la API pública** del juez (`judge.evaluate(case)`, `judge.compare(question, a, b, key=…)`):

- `verbosity_gain`: sobre los **100 primeros** casos con `truth["grounded"]`, `status == "ok"` y puntuación de relevancia del **juez de reglas** `RULE.evaluate(c).scores["relevance"].value ≤ 0,7`, la media de `relevance` del juez tras añadir `pad * 3` a la respuesta **menos** la media original.
- `position_consistency`: sobre los **100 primeros** casos con `truth["grounded"]`, `status == "ok"` y `n_tool_calls ≥ 1`, la fracción en que el juez elige **la misma respuesta** en ambos órdenes al comparar `a = c.answer` frente a `b = c.answer + "."` con `key=str(i)` (`i` = posición en esa lista). Elegir la misma respuesta significa `(compare(q, a, b) == "A") != (compare(q, b, a) == "A")`.

In [ ]:
PAD = " Gracias por contactar con SupportOps; estamos encantados de ayudarle con todo lo que necesite y quedamos a su entera disposición para cualquier otra consulta."

def bias_probe(judge, cases, pad) -> dict:
    # TODO 4
    raise NotImplementedError("TODO 4")

try:
    print(bias_probe(JUDGE, CASES, PAD))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 5 · ¿Cuánto me puedo fiar del juez?

**TODO 5.** `calibrate(judge, calib, prod)` devuelve un `dict`:

- `sens`, `spec`: sensibilidad y especificidad del juez para detectar **alucinaciones** (positivo = `groundedness` con etiqueta `fail`; verdad = `not c.truth["grounded"]`) sobre `calib`.
- `sens_ci`, `spec_ci`: intervalos de Wilson 95 % (`wilson_interval(k, n)`) de ambas.
- `observed`: tasa de `fail` de `groundedness` del juez sobre `prod`.
- `corrected`: `observed` corregida con Rogan–Gladen usando `sens` y `spec`.

In [ ]:
def calibrate(judge, calib, prod) -> dict:
    # TODO 5
    raise NotImplementedError("TODO 5")

try:
    CAL = calibrate(JUDGE, CALIB, PROD)
    print({k: (tuple(round(x, 3) for x in v) if isinstance(v, tuple) else round(v, 4)) for k, v in CAL.items()}, "· verdad en producción:", round(float(np.mean(TRUTH[200:])), 4))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 6 · Elegir la tasa de muestreo

**TODO 6.** `choose_rate(rates, judge_cost, agent_cost, n_day, max_overhead, p=0.85)` devuelve `{"rate", "overhead", "margin"}`: la **mayor** tasa de `rates` cuyo sobrecoste (coste diario del juez / coste diario del agente) no supera `max_overhead`, su sobrecoste y el margen de error del IC 95 % de la calidad
(`1,96·√(p(1−p)/n)` con `n = rate · n_day`). Si ninguna cumple, devuelve `{"rate": None, …}` con la menor tasa evaluada.

In [ ]:
def choose_rate(rates, judge_cost, agent_cost, n_day, max_overhead, p=0.85) -> dict:
    # TODO 6
    raise NotImplementedError("TODO 6")

try:
    _jc = float(np.mean([RULE.evaluate(c).cost_usd for c in CASES])); _ac = float(np.mean([c.cost_usd for c in CASES]))
    print("coste juez/agente por ticket:", round(_jc, 6), round(_ac, 6), "->", choose_rate([0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0], _jc, _ac, 20000, 0.5))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 7 · El panel de calidad como código

**TODO 7.** `build_quality_dashboard()` devuelve el `dict` de un dashboard (`Dashboard(...).to_dict()`, módulo 10) con **uid** `calidad-online`, título «14 · Calidad online (proyecto)» y **al menos 8 paneles con `description`**. Debe incluir estas consultas:

1. **Calidad estimada (post-estratificada)**: usa `supportops_eval_population_total` y `supportops_eval_results_total` con `by (supportops_eval_stratum)`.
2. **Calidad por versión**: `sum by (gen_ai_agent_version)` sobre `supportops_eval_results_total` con `gen_ai_evaluation_score_label="pass"`.
3. **Cobertura**: `supportops_eval_sampled_total` / `supportops_eval_population_total`.
4. **Distribución de puntuaciones**: heatmap de `supportops_eval_score_bucket`.
5. **Feedback**: `supportops_feedback_total`.
6. **Coste del juez**: `supportops_eval_cost_USD_total`.
7. **Evaluaciones fallidas (Loki)**: contiene `service_name="supportops-evaluator"` y `gen_ai_evaluation_score_label="fail"`.
8. **Trazas con fallo (Tempo)**: contiene `supportops.evaluate` y `span.supportops.eval.passed`.

<details>
<summary>🪜 Pista 1</summary>

Estructura (módulo 10): `d = Dashboard('calidad-online', '14 · Calidad online (proyecto)', tags=['curso', 'calidad'])`; `d.add(stat('…', 'promql', description='…'), timeseries('…', [prom('promql', '{{leyenda}}')], description='…'), heatmap('…', 'promql', description='…'), table('…', [loki('logql', '')], description='…'), traces_table('…', 'traceql', description='…'))`; devuelve `d.to_dict()`.

</details>

In [ ]:
def build_quality_dashboard() -> dict:
    # TODO 7
    raise NotImplementedError("TODO 7")

try:
    D = build_quality_dashboard(); print(len(D["panels"]), "paneles · lint:", lint(D) or "OK")
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 8 · Rúbrica

In [ ]:
REF = {"est": 0.845256794955425, "se": 0.027401340101016983, "gain": 0.21557142857142855, "pos": 0.2, "sens": 0.9333333333333333, "spec": 0.9891891891891892}   # calibrado con la solución (semilla 42)

def run_rubric():
    rows = []
    def add(r, d, ok, det=""): rows.append({"criterio": r, "descripción": d, "resultado": "✅ OK" if ok else "❌ FALLA", "detalle": det})
    # R1
    try:
        pred = [check_groundedness(c.answer, c.context, c.question)["score"] == 0.0 for c in CASES]
        cm = confusion(pred, TRUTH)
        ok = cm["sensitivity"] >= 0.98 and cm["specificity"] >= 0.98
        add("R1", "check_groundedness: sens. y espec. ≥ 0,98", ok, f"sens={cm['sensitivity']:.3f} espec={cm['specificity']:.3f} (tp={cm['tp']} fp={cm['fp']} fn={cm['fn']})")
    except Exception as e: add("R1", "check_groundedness", False, f"{type(e).__name__}: {e}")
    # R2
    try:
        est, se = stratified_estimate(EVALS, POP)
        ok = abs(est - REF["est"]) < 1e-9 and abs(se - REF["se"]) < 1e-9 and abs(est - FULL_PASS) < 0.03
        add("R2", "stratified_estimate: exacto y cerca de la verdad", ok, f"est={est:.6f} (ref {REF['est']:.6f}) se={se:.6f} (ref {REF['se']:.6f}) · verdad={FULL_PASS:.4f}")
    except Exception as e: add("R2", "stratified_estimate", False, f"{type(e).__name__}: {e}")
    # R3
    try:
        c0 = next(c for c in CASES if c.response_id)
        ea, ma = eval_telemetry(c0, "groundedness", 0.0, "fail", "x" * 500, "error", 1.0)
        ea2, _ = eval_telemetry(dataclasses.replace(c0, response_id=""), "relevance", 1, "pass", "", "base", 0.1)
        checks = {"evento válido": not validate_eval_event(ea) and not validate_eval_event(ea2), "score.value es float": isinstance(ea["gen_ai.evaluation.score.value"], float) and isinstance(ea2["gen_ai.evaluation.score.value"], float),
                  "explicación ≤ 300 y ausente si vacía": len(ea["gen_ai.evaluation.explanation"]) == 300 and "gen_ai.evaluation.explanation" not in ea2,
                  "response.id solo si existe": ea["gen_ai.response.id"] == c0.response_id and "gen_ai.response.id" not in ea2,
                  "etiquetas de métrica acotadas": set(ma) == {"gen_ai.evaluation.name", "gen_ai.evaluation.score.label", "supportops.eval.stratum", "gen_ai.agent.version"} and ea["supportops.ticket.id"] == c0.ticket_id}
        add("R3", "eval_telemetry: 5 comprobaciones", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R3", "eval_telemetry", False, f"{type(e).__name__}: {e}")
    # R4
    try:
        b = bias_probe(JUDGE, CASES, PAD)
        ok = abs(b["verbosity_gain"] - REF["gain"]) <= 0.02 and abs(b["position_consistency"] - REF["pos"]) <= 0.05
        add("R4", "bias_probe: ganancia por verbosidad y consistencia de posición", ok, f"ganancia={b['verbosity_gain']:.3f} (ref {REF['gain']:.3f}) · consistencia={b['position_consistency']:.2f} (ref {REF['pos']:.2f})")
    except Exception as e: add("R4", "bias_probe", False, f"{type(e).__name__}: {e}")
    # R5
    try:
        cal = calibrate(JUDGE, CALIB, PROD); truth_prod = float(np.mean(TRUTH[200:]))
        ok = abs(cal["sens"] - REF["sens"]) < 1e-9 and abs(cal["spec"] - REF["spec"]) < 1e-9 and cal["sens_ci"][0] < cal["sens"] < cal["sens_ci"][1] and abs(cal["corrected"] - truth_prod) <= 0.01 and abs(cal["corrected"] - truth_prod) < abs(cal["observed"] - truth_prod)
        add("R5", "calibrate: sens/espec exactas, IC y tasa corregida", ok, f"sens={cal['sens']:.3f} spec={cal['spec']:.3f} · observada={cal['observed']:.4f} corregida={cal['corrected']:.4f} verdad={truth_prod:.4f}")
    except Exception as e: add("R5", "calibrate", False, f"{type(e).__name__}: {e}")
    # R6
    try:
        r1 = choose_rate([0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0], 0.0027, 0.0003, 20000, 0.5)
        r2 = choose_rate([0.5, 1.0], 0.0027, 0.0003, 20000, 0.5)
        ok = r1["rate"] == 0.05 and abs(r1["overhead"] - 0.45) < 1e-9 and abs(r1["margin"] - 1.96 * math.sqrt(0.85 * 0.15 / 1000)) < 1e-9 and r2["rate"] is None
        add("R6", "choose_rate: presupuesto y margen", ok, f"{r1} · sin solución -> {r2}")
    except Exception as e: add("R6", "choose_rate", False, f"{type(e).__name__}: {e}")
    # R7
    try:
        import re as _re
        D = build_quality_dashboard()
        exprs = []
        for p in D["panels"]:
            for t in p.get("targets", []): exprs.append(t.get("expr") or t.get("query") or "")
        has = lambda *pats: any(all(_re.search(p, e) for p in pats) for e in exprs)
        checks = {"lint limpio, uid y ≥ 8 paneles": not lint(D) and D["uid"] == "calidad-online" and len(D["panels"]) >= 8,
                  "calidad post-estratificada": has(r"supportops_eval_population_total", r"supportops_eval_results_total", r"by \(supportops_eval_stratum\)"),
                  "calidad por versión": has(r"sum by \(gen_ai_agent_version\)", r"supportops_eval_results_total", r'score_label="pass"'), "cobertura": has(r"supportops_eval_sampled_total", r"supportops_eval_population_total"),
                  "distribución": has(r"supportops_eval_score_bucket"), "feedback": has(r"supportops_feedback_total"), "coste del juez": has(r"supportops_eval_cost_USD_total"),
                  "Loki fallidas": has(r'service_name="supportops-evaluator"', r'gen_ai_evaluation_score_label="fail"'), "Tempo": has(r"supportops\.evaluate", r"span\.supportops\.eval\.passed")}
        add("R7", "build_quality_dashboard: lint + 8 consultas", all(checks.values()), str({k: v for k, v in checks.items() if not v} or "todas OK"))
    except Exception as e: add("R7", "build_quality_dashboard", False, f"{type(e).__name__}: {e}")
    return pd.DataFrame(rows).set_index("criterio")

In [ ]:
# Ejecuta esta celda cuando hayas terminado los TODO
display(run_rubric())

---
## Paso final (opcional) · Míralo funcionar

Con el stack (`docker compose --profile core --profile metrics up -d`): guarda tu dashboard con `Dashboard.save(...)` en `stack/grafana/dashboards/`, y ejecuta `LivePipeline(("v1",), mode="otlp", sampler=StratifiedSampler()).run_live(generate_tickets(150, seed=42, duration_s=60, fault_rates=FAULTS, start_s=0.0), 60)`
(de `supportops.regression`). Espera ≈ 20 s y abre http://localhost:3000 (`admin`/`admin`): el panel «Calidad estimada» debe moverse y los logs de evaluaciones fallidas deben enlazar con la traza del agente.

## Retos extra (nivel experto)

1. **Cascada de evaluadores**: ejecuta primero las reglas y envía al juez solo los casos con `groundedness` incierto (0,3–0,7). ¿Cuánto coste ahorras y cuánta sensibilidad pierdes? Mídelo con la verdad.
2. **Juez por pares con ambos órdenes**: implementa `compare_both(judge, q, a, b)` que devuelve `"A"`, `"B"` o `"tie"` y mide cómo cambia la consistencia.
3. **IC de la calidad por versión**: con dos versiones del agente (módulo 15) calcula la diferencia de calidad con intervalo y decide si es significativa con el n evaluado.
4. **Calibración con deriva**: cambia el `MockJudge` a `miss_rate=0.25` y comprueba que `calibrate` lo detecta y que la tasa corregida se mantiene cerca de la verdad.
5. **Evaluación del feedback**: usa `simulate_feedback` para forzar la evaluación de todos los tickets con valoración negativa (se suman al estrato `error`); ¿sesga el estimador? ¿cómo lo corriges?

## Reflexión (producción e hilo conductor)

- Tu juez cuesta ~9× un ticket: ¿qué tasa de muestreo defenderías ante finanzas (módulo 13) y qué margen de error aceptas? ¿Cómo cambia la respuesta para un cliente premium?
- ¿Dónde pondrías el **umbral de alerta** de calidad para que no despierte a nadie por ruido del juez? (módulo 16: SLO de calidad y *min events*).
- ¿Qué parte de tu evaluador deberías poder **auditar** si un cliente reclama que su ticket fue evaluado con sus datos personales? (módulo 20).
- El módulo 15 usará los casos con evaluación fallida para construir el dataset de regresión: ¿qué campos de la telemetría necesitarías conservar?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable (redefine las funciones y vuelve a lanzar la rúbrica).

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
def check_groundedness(answer, context, question=""):
    claims = [("id", m) for m in re.findall(r"\b(?:ORD-\d{5}|TRK\d{6,}|RF-\d+|HO-\d+|KB-\d+)\b", answer) if m != "ORD-00000"]
    claims += [("eur", m) for m in re.findall(r"(\d+(?:[.,]\d+)?)\s?€", answer)]
    claims += [("num", f"{n} {u}") for n, u in re.findall(r"(\d+)\s+(años|unidades|horas|meses|días|dias)\b", answer)]
    nums = {round(float(x.replace(",", ".")), 2) for x in re.findall(r"\d+(?:[.,]\d+)?", context)}
    bad = []
    for kind, val in claims:
        if kind == "id": ok = val in context or val in question
        elif val in ("30 días", "5 días"): ok = True
        else: ok = round(float(val.split()[0].replace(",", ".")), 2) in nums
        if not ok: bad.append((kind, val))
    return {"score": 0.0 if bad else 1.0, "unsupported": bad}

def stratified_estimate(evals, population):
    N = sum(population.values()); est = var = 0.0
    for s, Ns in population.items():
        xs = [e["passed"] for e in evals if e["stratum"] == s]
        n = len(xs)
        if n == 0: continue
        p, w = sum(xs) / n, Ns / N
        est += w * p
        var += w * w * p * (1 - p) / max(n - 1, 1) * max(0.0, 1 - n / Ns)
    return est, math.sqrt(var)

def eval_telemetry(case, name, score, label, explanation, stratum, sample_rate):
    ev = {"gen_ai.evaluation.name": name, "gen_ai.evaluation.score.value": float(score), "gen_ai.evaluation.score.label": label}
    if explanation: ev["gen_ai.evaluation.explanation"] = explanation[:300]
    if case.response_id: ev["gen_ai.response.id"] = case.response_id
    ev.update({"supportops.ticket.id": case.ticket_id, "supportops.eval.stratum": stratum, "supportops.eval.sample_rate": float(sample_rate), "gen_ai.agent.version": case.version})
    mt = {"gen_ai.evaluation.name": name, "gen_ai.evaluation.score.label": label, "supportops.eval.stratum": stratum, "gen_ai.agent.version": case.version}
    return ev, mt

def bias_probe(judge, cases, pad):
    low = [c for c in cases if c.truth["grounded"] and c.status == "ok" and RULE.evaluate(c).scores["relevance"].value <= 0.7][:100]
    gain = float(np.mean([judge.evaluate(dataclasses.replace(c, answer=c.answer + pad * 3)).scores["relevance"].value - judge.evaluate(c).scores["relevance"].value for c in low]))
    pairs = [c for c in cases if c.truth["grounded"] and c.status == "ok" and c.n_tool_calls >= 1][:100]
    same = 0
    for i, c in enumerate(pairs):
        a, b = c.answer, c.answer + "."
        same += (judge.compare(c.question, a, b, key=str(i)) == "A") != (judge.compare(c.question, b, a, key=str(i)) == "A")
    return {"verbosity_gain": gain, "position_consistency": same / len(pairs)}

def calibrate(judge, calib, prod):
    fails = lambda cs: [judge.evaluate(c).scores["groundedness"].label == "fail" for c in cs]
    pred, truth = fails(calib), [not c.truth["grounded"] for c in calib]
    cm = confusion(pred, truth)
    sens, spec = cm["sensitivity"], cm["specificity"]
    observed = float(np.mean(fails(prod)))
    return {"sens": sens, "spec": spec, "sens_ci": wilson_interval(cm["tp"], cm["tp"] + cm["fn"]), "spec_ci": wilson_interval(cm["tn"], cm["tn"] + cm["fp"]),
            "observed": observed, "corrected": rogan_gladen(observed, sens, spec)}

def choose_rate(rates, judge_cost, agent_cost, n_day, max_overhead, p=0.85):
    best = None
    for r in sorted(rates):
        over = r * n_day * judge_cost / (n_day * agent_cost)
        if over <= max_overhead + 1e-12: best = (r, over)
    if best is None:
        r = min(rates); return {"rate": None, "overhead": r * judge_cost / agent_cost, "margin": 1.96 * math.sqrt(p * (1 - p) / (r * n_day))}
    r, over = best
    return {"rate": r, "overhead": over, "margin": 1.96 * math.sqrt(p * (1 - p) / (r * n_day))}

def build_quality_dashboard():
    d = Dashboard("calidad-online", "14 · Calidad online (proyecto)", tags=["curso", "calidad"], description="Calidad de las respuestas de SupportOps medida online.")
    pop = "sum by (supportops_eval_stratum) (rate(supportops_eval_population_total[5m]))"
    ok_ = 'sum by (supportops_eval_stratum) (rate(supportops_eval_results_total{gen_ai_evaluation_name="groundedness",gen_ai_evaluation_score_label="pass"}[5m]))'
    all_ = 'sum by (supportops_eval_stratum) (rate(supportops_eval_results_total{gen_ai_evaluation_name="groundedness"}[5m]))'
    d.add(
        stat("Calidad estimada (post-estratificada)", f"sum({pop} * ({ok_} / {all_})) / sum(rate(supportops_eval_population_total[5m]))", unit="percentunit", w=6, description="Tasa de respuestas fundamentadas, ponderada por el tamaño real de cada estrato."),
        timeseries("Calidad por versión", [prom('sum by (gen_ai_agent_version) (rate(supportops_eval_results_total{gen_ai_evaluation_name="groundedness",gen_ai_evaluation_score_label="pass"}[5m])) / sum by (gen_ai_agent_version) (rate(supportops_eval_results_total{gen_ai_evaluation_name="groundedness"}[5m]))', "{{gen_ai_agent_version}}")], unit="percentunit", w=9, description="Groundedness por versión del agente."),
        timeseries("Cobertura de la evaluación", [prom("sum(rate(supportops_eval_sampled_total[5m])) / sum(rate(supportops_eval_population_total[5m]))", "evaluadas / total")], unit="percentunit", w=9, description="Fracción de peticiones evaluadas."),
        heatmap("Distribución de puntuaciones", "sum by (le) (increase(supportops_eval_score_bucket[$__rate_interval]))", unit="short", w=12, description="Histograma de puntuaciones 0..1 de los jueces."),
        timeseries("Feedback de usuarios", [prom("sum by (supportops_feedback_rating) (rate(supportops_feedback_total[5m]))", "{{supportops_feedback_rating}}")], w=12, description="Valoraciones explícitas (sintéticas)."),
        timeseries("Coste del juez", [prom("sum(rate(supportops_eval_cost_USD_total[5m]))", "juez")], unit="currencyUSD", w=8, description="Gasto del juez (precios ilustrativos)."),
        logs("Evaluaciones fallidas (Loki)", '{service_name="supportops-evaluator"} | gen_ai_evaluation_score_label="fail"', w=8, description="Evaluaciones con etiqueta fail."),
        traces_table("Trazas de evaluación con fallo (Tempo)", '{ name = "supportops.evaluate" && span.supportops.eval.passed = false }', w=8, description="Evaluaciones que no pasaron."),
    )
    return d.to_dict()

display(run_rubric())